# t-SNE (t-Distributed Stochastic Neighbor Embedding)

A comprehensive guide to understanding and implementing t-SNE for dimensionality reduction and visualization.

## Table of Contents

1. [Theory](#1.-Theory)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

---

## 1. Theory

### 1.1 Stochastic Neighbor Embedding (SNE) Concept

t-SNE is a nonlinear dimensionality reduction technique that is particularly well-suited for embedding high-dimensional data into a 2D or 3D space for visualization.

**Core Idea:** Convert high-dimensional Euclidean distances between points into conditional probabilities that represent similarities.

#### High-Dimensional Space (Input)

For each pair of points $x_i$ and $x_j$, we compute a conditional probability $p_{j|i}$ that represents the probability that $x_i$ would pick $x_j$ as its neighbor if neighbors were picked in proportion to their probability density under a Gaussian centered at $x_i$:

$$p_{j|i} = \frac{\exp(-||x_i - x_j||^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-||x_i - x_k||^2 / 2\sigma_i^2)}$$

The joint probability is symmetrized: $p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$

#### Low-Dimensional Space (Output)

In the low-dimensional map, we use a **Student's t-distribution** (with one degree of freedom) instead of a Gaussian:

$$q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum_{k \neq l} (1 + ||y_k - y_l||^2)^{-1}}$$

### 1.2 Why Student's t-Distribution?

The **crowding problem**: In high dimensions, the volume of a sphere grows exponentially with dimension. When we try to faithfully represent moderate distances in high dimensions, points get "crowded" in low dimensions.

**Solution:** Use a heavy-tailed t-distribution in the low-dimensional space. This allows:
- Nearby points to remain close
- Moderate-distance points to be pushed further apart
- Better separation of clusters

### 1.3 Perplexity Parameter

Perplexity is a smooth measure of the effective number of neighbors:

$$Perp(P_i) = 2^{H(P_i)}$$

where $H(P_i)$ is the Shannon entropy of $P_i$:

$$H(P_i) = -\sum_j p_{j|i} \log_2 p_{j|i}$$

**Typical values:** 5-50 (commonly 30)

- **Low perplexity:** Focus on local structure, small neighborhoods
- **High perplexity:** Consider broader structure, larger neighborhoods

### 1.4 KL Divergence Optimization

t-SNE minimizes the Kullback-Leibler divergence between the high-dimensional probability distribution $P$ and the low-dimensional distribution $Q$:

$$C = KL(P||Q) = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

The gradient is:

$$\frac{\partial C}{\partial y_i} = 4 \sum_j (p_{ij} - q_{ij})(y_i - y_j)(1 + ||y_i - y_j||^2)^{-1}$$

### 1.5 Time and Space Complexity

| Aspect | Complexity | Notes |
|--------|------------|-------|
| **Time** | $O(n^2)$ | Computing pairwise distances and gradients |
| **Space** | $O(n^2)$ | Storing pairwise probability matrices |

**Optimization:** Barnes-Hut approximation reduces time to $O(n \log n)$ but our educational implementation uses the exact $O(n^2)$ method.

### 1.6 Important Limitation: No Transform for New Data

t-SNE does **NOT** support projecting new data points. Unlike PCA which learns a linear transformation, t-SNE:
- Learns positions for specific data points
- Has no parametric mapping function
- Requires re-running on the entire dataset if new points are added

---

## 2. Implementation from Scratch

A simplified t-SNE implementation using NumPy for educational purposes.

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE as SklearnTSNE
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
class TSNE:
    """
    t-Distributed Stochastic Neighbor Embedding (t-SNE)
    
    A simplified implementation for educational purposes.
    Real implementations use Barnes-Hut optimization for O(n log n) complexity.
    
    Parameters
    ----------
    n_components : int, default=2
        Dimension of the embedded space (typically 2 or 3 for visualization)
    perplexity : float, default=30.0
        Related to the number of nearest neighbors. Typical range: 5-50
    n_iter : int, default=1000
        Number of iterations for optimization
    learning_rate : float, default=200.0
        Learning rate for gradient descent
    momentum : float, default=0.8
        Momentum for gradient descent updates
    early_exaggeration : float, default=12.0
        Exaggeration factor for early iterations
    n_iter_early_exag : int, default=250
        Number of iterations with early exaggeration
    random_state : int, default=None
        Random seed for reproducibility
    """
    
    def __init__(self, n_components=2, perplexity=30.0, n_iter=1000, 
                 learning_rate=200.0, momentum=0.8, early_exaggeration=12.0,
                 n_iter_early_exag=250, random_state=None):
        self.n_components = n_components
        self.perplexity = perplexity
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.early_exaggeration = early_exaggeration
        self.n_iter_early_exag = n_iter_early_exag
        self.random_state = random_state
        
        # Store KL divergence history for diagnostics
        self.kl_divergence_history_ = []
        
    def _compute_pairwise_distances(self, X):
        """Compute squared Euclidean distances between all pairs of points."""
        sum_X = np.sum(X ** 2, axis=1)
        # Using ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a.b
        D = sum_X[:, np.newaxis] + sum_X[np.newaxis, :] - 2 * np.dot(X, X.T)
        return np.maximum(D, 0)  # Ensure non-negative
    
    def _binary_search_perplexity(self, distances, target_perplexity, tol=1e-5, max_iter=50):
        """
        Perform binary search to find sigma values that achieve target perplexity.
        
        Parameters
        ----------
        distances : ndarray of shape (n_samples, n_samples)
            Squared pairwise distances
        target_perplexity : float
            Desired perplexity value
        
        Returns
        -------
        P : ndarray of shape (n_samples, n_samples)
            Conditional probability matrix
        """
        n_samples = distances.shape[0]
        P = np.zeros((n_samples, n_samples))
        target_entropy = np.log(target_perplexity)
        
        for i in range(n_samples):
            # Binary search for sigma_i
            beta_min, beta_max = -np.inf, np.inf
            beta = 1.0  # beta = 1 / (2 * sigma^2)
            
            # Get distances from point i to all other points
            Di = distances[i, np.concatenate([np.arange(i), np.arange(i+1, n_samples)])]
            
            for _ in range(max_iter):
                # Compute conditional probabilities
                P_i = np.exp(-Di * beta)
                sum_P_i = np.sum(P_i)
                
                if sum_P_i == 0:
                    sum_P_i = 1e-10
                
                P_i = P_i / sum_P_i
                
                # Compute entropy
                H = -np.sum(P_i * np.log(P_i + 1e-10))
                
                # Check for convergence
                H_diff = H - target_entropy
                if np.abs(H_diff) < tol:
                    break
                
                # Binary search update
                if H_diff > 0:
                    beta_min = beta
                    beta = beta * 2 if beta_max == np.inf else (beta + beta_max) / 2
                else:
                    beta_max = beta
                    beta = beta / 2 if beta_min == -np.inf else (beta + beta_min) / 2
            
            # Fill in the probability matrix
            P[i, np.concatenate([np.arange(i), np.arange(i+1, n_samples)])] = P_i
        
        return P
    
    def _compute_joint_probabilities(self, X):
        """
        Compute joint probability matrix P from input data.
        
        Symmetrizes the conditional probabilities: P = (P + P^T) / (2n)
        """
        distances = self._compute_pairwise_distances(X)
        P = self._binary_search_perplexity(distances, self.perplexity)
        
        # Symmetrize
        P = (P + P.T) / (2 * X.shape[0])
        
        # Ensure minimum probability to avoid numerical issues
        P = np.maximum(P, 1e-12)
        
        return P
    
    def _compute_low_dim_affinities(self, Y):
        """
        Compute low-dimensional affinities Q using Student's t-distribution.
        
        q_ij = (1 + ||y_i - y_j||^2)^-1 / sum_k!=l (1 + ||y_k - y_l||^2)^-1
        """
        distances = self._compute_pairwise_distances(Y)
        
        # Student's t-distribution with 1 degree of freedom
        Q = 1 / (1 + distances)
        np.fill_diagonal(Q, 0)
        
        # Normalize
        Q = Q / np.sum(Q)
        Q = np.maximum(Q, 1e-12)
        
        return Q, distances
    
    def _compute_gradient(self, P, Q, Y, distances):
        """
        Compute the gradient of the KL divergence.
        
        dC/dy_i = 4 * sum_j (p_ij - q_ij)(y_i - y_j)(1 + ||y_i - y_j||^2)^-1
        """
        n_samples = Y.shape[0]
        
        # Compute (p_ij - q_ij) * (1 + ||y_i - y_j||^2)^-1
        PQ_diff = P - Q
        inv_distances = 1 / (1 + distances)
        np.fill_diagonal(inv_distances, 0)
        
        # Compute gradient
        grad = np.zeros_like(Y)
        for i in range(n_samples):
            diff = Y[i] - Y
            grad[i] = 4 * np.sum((PQ_diff[i] * inv_distances[i])[:, np.newaxis] * diff, axis=0)
        
        return grad
    
    def _compute_kl_divergence(self, P, Q):
        """Compute the KL divergence between P and Q."""
        return np.sum(P * np.log(P / Q))
    
    def fit_transform(self, X):
        """
        Fit t-SNE and return the embedded coordinates.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Input data
        
        Returns
        -------
        Y : ndarray of shape (n_samples, n_components)
            Embedded coordinates
        """
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        n_samples = X.shape[0]
        
        # Step 1: Compute joint probabilities P in high-dimensional space
        print("Computing pairwise similarities...")
        P = self._compute_joint_probabilities(X)
        
        # Step 2: Initialize low-dimensional embeddings randomly
        Y = np.random.randn(n_samples, self.n_components) * 1e-4
        
        # Initialize velocity for momentum
        velocity = np.zeros_like(Y)
        
        # Reset KL divergence history
        self.kl_divergence_history_ = []
        
        # Store intermediate embeddings for visualization
        self.embeddings_history_ = []
        
        print("Optimizing...")
        
        # Step 3: Gradient descent optimization
        for iteration in range(self.n_iter):
            # Apply early exaggeration
            if iteration < self.n_iter_early_exag:
                P_exag = P * self.early_exaggeration
            else:
                P_exag = P
            
            # Compute low-dimensional affinities
            Q, distances = self._compute_low_dim_affinities(Y)
            
            # Compute gradient
            grad = self._compute_gradient(P_exag, Q, Y, distances)
            
            # Update with momentum
            velocity = self.momentum * velocity - self.learning_rate * grad
            Y = Y + velocity
            
            # Center the solution
            Y = Y - np.mean(Y, axis=0)
            
            # Compute and store KL divergence
            kl_div = self._compute_kl_divergence(P, Q)
            self.kl_divergence_history_.append(kl_div)
            
            # Store embeddings at certain iterations for visualization
            if iteration in [0, 50, 100, 250, 500, 750, 999]:
                self.embeddings_history_.append((iteration, Y.copy()))
            
            # Print progress
            if (iteration + 1) % 100 == 0:
                print(f"Iteration {iteration + 1}/{self.n_iter}, KL divergence: {kl_div:.4f}")
        
        self.embedding_ = Y
        self.kl_divergence_ = self.kl_divergence_history_[-1]
        
        return Y

---

## 3. Training & Optimization

Training t-SNE on a subset of the digits dataset.

In [ ]:
# Load the digits dataset
digits = load_digits()
X_full = digits.data
y_full = digits.target

print(f"Full dataset shape: {X_full.shape}")
print(f"Number of classes: {len(np.unique(y_full))}")
print(f"Features per sample: {X_full.shape[1]} (8x8 pixels)")

In [ ]:
# Use a subset for CPU efficiency (O(n^2) complexity)
n_samples = 500

# Stratified sampling to maintain class balance
np.random.seed(42)
indices = []
for digit in range(10):
    digit_indices = np.where(y_full == digit)[0]
    selected = np.random.choice(digit_indices, size=n_samples // 10, replace=False)
    indices.extend(selected)

indices = np.array(indices)
np.random.shuffle(indices)

X = X_full[indices]
y = y_full[indices]

print(f"Working dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Visualize some sample digits
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'{y[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits from Dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Train our t-SNE implementation
print("Training custom t-SNE implementation...")
print("="*50)

tsne = TSNE(
    n_components=2,
    perplexity=30.0,
    n_iter=1000,
    learning_rate=200.0,
    random_state=42
)

Y = tsne.fit_transform(X)

print("="*50)
print(f"Final KL divergence: {tsne.kl_divergence_:.4f}")
print(f"Embedding shape: {Y.shape}")

---

## 4. Diagnostics & Evaluation

Monitoring the optimization process and understanding hyperparameter effects.

In [ ]:
# Plot KL divergence during optimization
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(tsne.kl_divergence_history_, 'b-', linewidth=1.5)
ax.axvline(x=250, color='r', linestyle='--', label='End of early exaggeration')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('KL Divergence', fontsize=12)
ax.set_title('KL Divergence During t-SNE Optimization', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Add inset for zoomed view of later iterations
ax_inset = ax.inset_axes([0.5, 0.4, 0.45, 0.45])
ax_inset.plot(tsne.kl_divergence_history_[300:], 'b-', linewidth=1.5)
ax_inset.set_xlabel('Iteration (from 300)', fontsize=9)
ax_inset.set_ylabel('KL Divergence', fontsize=9)
ax_inset.set_title('Zoomed View', fontsize=10)
ax_inset.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Study the effect of perplexity
perplexities = [5, 15, 30, 50]
embeddings_by_perplexity = {}

print("Comparing different perplexity values...")
print("="*50)

# Use smaller sample for perplexity comparison (faster)
n_perp_samples = 300
X_perp = X[:n_perp_samples]
y_perp = y[:n_perp_samples]

for perp in perplexities:
    print(f"\nPerplexity = {perp}")
    tsne_perp = TSNE(
        n_components=2,
        perplexity=perp,
        n_iter=750,
        learning_rate=200.0,
        random_state=42
    )
    embeddings_by_perplexity[perp] = tsne_perp.fit_transform(X_perp)

print("\n" + "="*50)
print("Perplexity comparison complete!")

---

## 5. Visualizations

Visualizing the t-SNE embeddings and comparing different configurations.

In [ ]:
# Create a nice color palette for digits 0-9
colors = plt.cm.tab10(np.linspace(0, 1, 10))

def plot_embedding(Y, y, title, ax=None):
    """Helper function to plot t-SNE embedding."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    for digit in range(10):
        mask = y == digit
        ax.scatter(Y[mask, 0], Y[mask, 1], c=[colors[digit]], 
                   label=str(digit), alpha=0.7, s=30, edgecolors='white', linewidth=0.5)
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('t-SNE 1', fontsize=11)
    ax.set_ylabel('t-SNE 2', fontsize=11)
    ax.legend(title='Digit', loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    return ax

In [ ]:
# Main t-SNE visualization
fig, ax = plt.subplots(figsize=(12, 10))
plot_embedding(Y, y, 't-SNE Embedding of Digits Dataset (Custom Implementation)', ax)
plt.tight_layout()
plt.show()

In [ ]:
# Perplexity comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, perp in zip(axes.flat, perplexities):
    Y_perp = embeddings_by_perplexity[perp]
    plot_embedding(Y_perp, y_perp, f'Perplexity = {perp}', ax)

plt.suptitle('Effect of Perplexity on t-SNE Embedding', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize iteration progression
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for ax, (iteration, Y_iter) in zip(axes.flat, tsne.embeddings_history_):
    for digit in range(10):
        mask = y == digit
        ax.scatter(Y_iter[mask, 0], Y_iter[mask, 1], c=[colors[digit]], 
                   alpha=0.7, s=20, edgecolors='white', linewidth=0.3)
    ax.set_title(f'Iteration {iteration + 1}', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

# Hide the last subplot if not used
if len(tsne.embeddings_history_) < 8:
    axes.flat[-1].axis('off')

plt.suptitle('t-SNE Optimization Progression', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Create an annotated version with some digit images
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fig, ax = plt.subplots(figsize=(14, 12))

# Plot all points
for digit in range(10):
    mask = y == digit
    ax.scatter(Y[mask, 0], Y[mask, 1], c=[colors[digit]], 
               label=str(digit), alpha=0.5, s=20)

# Add some digit images as annotations
np.random.seed(42)
sample_indices = np.random.choice(len(X), 30, replace=False)

for idx in sample_indices:
    img = X[idx].reshape(8, 8)
    imagebox = OffsetImage(img, zoom=1.5, cmap='gray')
    ab = AnnotationBbox(imagebox, (Y[idx, 0], Y[idx, 1]),
                        frameon=True, pad=0.1,
                        bboxprops=dict(edgecolor=colors[y[idx]], linewidth=2))
    ax.add_artist(ab)

ax.set_title('t-SNE Embedding with Sample Digit Images', fontsize=14)
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.legend(title='Digit', loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. Use Cases & Guidelines

### When to Use t-SNE

| Use Case | Description |
|----------|-------------|
| **Visualization** | Primary use: visualizing high-dimensional data in 2D or 3D |
| **Cluster Exploration** | Discovering natural groupings in data |
| **Quality Assessment** | Checking if learned representations capture meaningful structure |
| **Anomaly Detection** | Identifying outliers that don't belong to any cluster |
| **High-dim to 2D/3D** | When the goal is specifically 2D or 3D embedding |

### When NOT to Use t-SNE

| Scenario | Why | Alternative |
|----------|-----|-------------|
| **Large datasets** (>10k samples) | O(n^2) complexity, very slow | Use UMAP or Barnes-Hut t-SNE |
| **New data projection** | No transform() method | Use PCA, UMAP, or autoencoders |
| **Preserving distances** | t-SNE preserves local, not global structure | Use MDS or UMAP |
| **Preprocessing for ML** | Non-deterministic, no inverse transform | Use PCA or autoencoders |
| **Interpretable dimensions** | Components have no meaning | Use PCA or NMF |
| **Real-time applications** | Too slow for inference | Use parametric methods |

### Perplexity Selection Guidelines

| Data Size | Recommended Perplexity | Notes |
|-----------|------------------------|-------|
| < 100 samples | 5-10 | Very small datasets need small perplexity |
| 100-500 samples | 10-30 | Balance local and global structure |
| 500-2000 samples | 30-50 | Standard range |
| > 2000 samples | 30-100 | Larger datasets can use higher perplexity |

**Rule of thumb:** Perplexity should be smaller than the number of points. Typical range is 5-50.

### Important Caveats

1. **Cluster sizes are not meaningful** - t-SNE can expand or contract clusters
2. **Distances between clusters are not meaningful** - Global structure is not preserved
3. **Multiple runs give different results** - Use random_state for reproducibility
4. **Can create false patterns** - Always validate findings with other methods

In [ ]:
# Demonstrate the stochasticity of t-SNE
print("Demonstrating t-SNE stochasticity (different random seeds)...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

X_demo = X[:200]  # Smaller subset for speed
y_demo = y[:200]

for ax, seed in zip(axes, [42, 123, 456]):
    tsne_demo = TSNE(n_components=2, perplexity=20, n_iter=500, random_state=seed)
    Y_demo = tsne_demo.fit_transform(X_demo)
    
    for digit in range(10):
        mask = y_demo == digit
        ax.scatter(Y_demo[mask, 0], Y_demo[mask, 1], c=[colors[digit]], 
                   alpha=0.7, s=30, label=str(digit))
    ax.set_title(f'Random Seed = {seed}', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

axes[0].legend(title='Digit', loc='best', fontsize=8)
plt.suptitle('t-SNE Results with Different Random Seeds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 7. Comparison with sklearn

Comparing our implementation with scikit-learn's optimized t-SNE.

In [ ]:
import time

# Prepare data for comparison
X_compare = X[:400]
y_compare = y[:400]

print("Comparing custom implementation with sklearn...")
print(f"Dataset size: {X_compare.shape[0]} samples")
print("="*60)

In [ ]:
# Our implementation
print("\n1. Custom Implementation:")
start_time = time.time()

tsne_custom = TSNE(
    n_components=2,
    perplexity=30.0,
    n_iter=1000,
    learning_rate=200.0,
    random_state=42
)
Y_custom = tsne_custom.fit_transform(X_compare)

custom_time = time.time() - start_time
print(f"Time: {custom_time:.2f} seconds")
print(f"Final KL divergence: {tsne_custom.kl_divergence_:.4f}")

In [ ]:
# sklearn implementation
print("\n2. sklearn Implementation:")
start_time = time.time()

tsne_sklearn = SklearnTSNE(
    n_components=2,
    perplexity=30.0,
    n_iter=1000,
    learning_rate='auto',
    init='random',
    random_state=42
)
Y_sklearn = tsne_sklearn.fit_transform(X_compare)

sklearn_time = time.time() - start_time
print(f"Time: {sklearn_time:.2f} seconds")
print(f"Final KL divergence: {tsne_sklearn.kl_divergence_:.4f}")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Custom implementation
for digit in range(10):
    mask = y_compare == digit
    axes[0].scatter(Y_custom[mask, 0], Y_custom[mask, 1], c=[colors[digit]], 
                    label=str(digit), alpha=0.7, s=40, edgecolors='white', linewidth=0.5)
axes[0].set_title(f'Custom t-SNE Implementation\nTime: {custom_time:.2f}s, KL: {tsne_custom.kl_divergence_:.4f}', fontsize=12)
axes[0].set_xlabel('t-SNE 1', fontsize=11)
axes[0].set_ylabel('t-SNE 2', fontsize=11)
axes[0].legend(title='Digit', loc='best')
axes[0].grid(True, alpha=0.3)

# sklearn implementation
for digit in range(10):
    mask = y_compare == digit
    axes[1].scatter(Y_sklearn[mask, 0], Y_sklearn[mask, 1], c=[colors[digit]], 
                    label=str(digit), alpha=0.7, s=40, edgecolors='white', linewidth=0.5)
axes[1].set_title(f'sklearn t-SNE Implementation\nTime: {sklearn_time:.2f}s, KL: {tsne_sklearn.kl_divergence_:.4f}', fontsize=12)
axes[1].set_xlabel('t-SNE 1', fontsize=11)
axes[1].set_ylabel('t-SNE 2', fontsize=11)
axes[1].legend(title='Digit', loc='best')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparison: Custom vs sklearn t-SNE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"{'Metric':<30} {'Custom':<15} {'sklearn':<15}")
print("-"*60)
print(f"{'Execution Time (s)':<30} {custom_time:<15.2f} {sklearn_time:<15.2f}")
print(f"{'KL Divergence':<30} {tsne_custom.kl_divergence_:<15.4f} {tsne_sklearn.kl_divergence_:<15.4f}")
print(f"{'Speed Ratio':<30} {custom_time/sklearn_time:<15.2f} {'1.00':<15}")
print("="*60)
print("\nNotes:")
print("- sklearn uses Barnes-Hut approximation for O(n log n) complexity")
print("- Our implementation uses exact O(n^2) for educational purposes")
print("- Both achieve similar quality embeddings")
print("- sklearn is optimized with Cython and parallel processing")

---

## Summary

### Key Takeaways

1. **t-SNE is for visualization**, not general dimensionality reduction
2. **Perplexity** controls the balance between local and global structure (typical: 5-50)
3. **O(n^2) complexity** limits usage to small/medium datasets (or use Barnes-Hut)
4. **No transform for new data** - must re-run on entire dataset
5. **Results are stochastic** - use random_state for reproducibility
6. **Don't over-interpret** - cluster sizes and distances are not meaningful

### Algorithm Steps

1. Compute pairwise affinities in high-dimensional space (Gaussian)
2. Initialize low-dimensional embeddings randomly
3. Compute affinities in low-dimensional space (Student's t-distribution)
4. Minimize KL divergence using gradient descent
5. Apply early exaggeration for better cluster separation

### Recommended Practices

- Run multiple times with different perplexities
- Try different random seeds to verify cluster stability
- Use PCA for preprocessing if data is very high-dimensional
- For large datasets, use sklearn with method='barnes_hut' or consider UMAP